# PIP: Poverty by Country and Year

This notebook downloads all available country-level PIP observations, across all available years. It keeps the survey, welfare, coverage, and poverty-line dimensions so observations are not incorrectly combined.

## 1. Import Libraries

In [1]:
import pandas as pd
import requests

## 2. Configure the PIP API

The query below requests every available country and year. PIP may return multiple observations for the same country and year because welfare type, survey coverage, reporting level, and poverty line differ.

In [2]:
PIP_URL = "https://api.worldbank.org/pip/v1/pip"

REQUEST_PARAMS = {
    "country": "ALL",
    "format": "json",
}

REQUEST_HEADERS = {"Accept": "application/json"}
TIMEOUT_SECONDS = 60
print(PIP_URL)
print(REQUEST_PARAMS)

https://api.worldbank.org/pip/v1/pip
{'country': 'ALL', 'format': 'json'}


## 3. Download All PIP Observations

In [3]:
response = requests.get(
    PIP_URL,
    params=REQUEST_PARAMS,
    headers=REQUEST_HEADERS,
    timeout=TIMEOUT_SECONDS,
)

print("Request URL:", response.url)
print("Status code:", response.status_code)
response.raise_for_status()

pip_df = pd.DataFrame(response.json())
print(f"Rows: {pip_df.shape[0]:,}")
print(f"Columns: {pip_df.shape[1]:,}")
pip_df.head()

Request URL: https://api.worldbank.org/pip/v1/pip?country=ALL&format=json
Status code: 200
Rows: 2,588
Columns: 44


,region_name,region_code,country_name,country_code,reporting_year,reporting_level,survey_acronym,survey_coverage,survey_year,welfare_type,...,reporting_pop,reporting_gdp,reporting_pce,is_interpolated,distribution_type,estimation_type,spl,spr,pg,estimate_type
0,Sub-Saharan Africa,SSF,Angola,AGO,2000,national,HBS,national,2000.21,consumption,...,16310860.19,2172.4633,NaN,False,micro,survey,3.950,0.3715,9.6092,None
1,Sub-Saharan Africa,SSF,Angola,AGO,2008,national,IBEP-MICS,national,2008.50,consumption,...,21996714.00,3559.1952,1748.4284,False,micro,survey,3.967,0.3534,6.8388,None
2,Sub-Saharan Africa,SSF,Angola,AGO,2018,national,IDREA,national,2018.17,consumption,...,31480496.09,3196.7901,2261.4557,False,micro,survey,3.240,0.4216,10.9642,None
3,Europe & Central Asia,ECS,Albania,ALB,1996,national,EWS,national,1996.00,consumption,...,3168033.00,1683.7697,1522.8719,False,micro,survey,5.010,0.2039,4.2094,None
4,Europe & Central Asia,ECS,Albania,ALB,2002,national,LSMS,national,2002.00,consumption,...,3051010.00,2297.1085,1572.2955,False,micro,survey,4.859,0.2244,4.4293,None


## 4. Create the Country-Year Poverty Table

`headcount` is the poverty rate as a proportion. The notebook converts it to a percentage and estimates the number of people below the selected poverty line using `reporting_pop`.

In [5]:
numeric_columns = [
    "reporting_year",
    "survey_year",
    "poverty_line",
    "headcount",
    "poverty_gap",
    "poverty_severity",
    "reporting_pop",
]

for column in numeric_columns:
    if column in pip_df.columns:
        pip_df[column] = pd.to_numeric(pip_df[column], errors="coerce")

pip_df["headcount_percent"] = pip_df["headcount"] * 100
pip_df["estimated_poor_population"] = pip_df["headcount"] * pip_df["reporting_pop"]

poverty_by_country_year = pip_df[
    [
        "country_code",
        "country_name",
        "reporting_year",
        "poverty_line",
        "welfare_type",
        "reporting_level",
        "survey_coverage",
        "headcount_percent",
        "estimated_poor_population",
        "poverty_gap",
        "poverty_severity",
        "survey_acronym",
    ]
].sort_values(["country_code", "reporting_year", "poverty_line"])

print("Countries:", poverty_by_country_year["country_code"].nunique())
print("Years:", poverty_by_country_year["reporting_year"].min(), "to", poverty_by_country_year["reporting_year"].max())
print("Rows in country-year table:", len(poverty_by_country_year))
poverty_by_country_year.head(20)

poverty_by_country_year.shape

Countries: 172
Years: 1963 to 2025
Rows in country-year table: 2588


(2588, 12)